<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
  <h1 style='color: #ffffff; margin: 0 0 8px 0; font-size: 2.5em;'>🎨 Fibo 2.0 Image Studio</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Free · Kaggle T4 x2 Edition</h3>
  <p style='color: #aaa; margin: 0; text-align: center;'>Text-to-Image • Image Editing • 4K Upscale (Real-ESRGAN + HYPIR) | BRIA Fibo 2.0 on ComfyUI</p>
</div>

<p align="center">
  <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white" /></a>
  <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white" /></a>
  <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white" /></a>
  <a href="https://github.com/cafermutluozkan"><img src="https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white" /></a>
</p>

---

### 🚀 What is Fibo 2.0?

**Fibo 2.0** is BRIA AI's 8B-parameter image model that was trained on **long structured JSON captions** instead of short prompts. You can describe every object, the lighting, the camera, the style and even the **exact text to render** field by field, and the model follows it closely. It also **edits images** from a plain-English instruction and can combine up to several reference images.

This notebook runs it on Kaggle's free **2× T4** GPUs, **without owning a GPU**. When it finishes, it opens a web page: type a prompt, press Generate, download the image.

| Feature | Detail |
|---|---|
| **Model** | Fibo 2.0 Dev (8B, int8) + Fibo 2.0 Edit Turbo (4-step editing) |
| **Text encoder** | Qwen3-VL-4B (int8) |
| **Output** | ~1 MP native (8 trained aspect ratios), **2K / 4K** with two upscalers: **⚡ Real-ESRGAN** (seconds) or **✨ HYPIR** (SIGGRAPH Asia 2025, adds real detail) |
| **GPU** | Kaggle T4 x2: the model runs on **both GPUs at once** (CFG split), text encoder + VAE on GPU 1, HYPIR in its own process on GPU 1 |
| **Speed** | Fast int8 kernels (PyTorch cu130 + comfy-kitchen), no previews, expandable CUDA memory |
| **Engine** | ComfyUI (pinned commit), every file pinned by revision + SHA256 |
| **License** | Fibo 2.0: CC BY-NC 4.0 (**non-commercial**) · details at the end |

### ⚡ Quick Start
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. Run **Steps 1-3** = settings, install, model download (~26 GB, ~10-15 min)
4. Run **Step 4** = launch the studio and open the **public link** (`https://....gradio.live`)

> 💡 **Tips:** Fibo listens best to JSON prompts (press **📋 Insert JSON template**). Start with **⚡ Fast** quality to test a prompt, then switch to **💎 Best**. The first image is slower because the models are loading.

---
## Step 1: Settings

Versions and files are pinned (commit, revision and SHA256). You normally do not need to change anything.

In [ ]:
# Step 1 - Settings. You normally do not need to change anything here.
# Every version and file is PINNED (commit / revision / SHA256), so "it worked yesterday, broken today" does not happen.
from pathlib import Path

COMFYUI_REPO = "https://github.com/comfy-org/ComfyUI.git"
COMFYUI_COMMIT = "1a82e6d01f395b43275c77acd289af1eb7090bb9"   # ComfyUI master 2026-10-22 (first builds with Fibo 2.0)
PORT = 8188

UPGRADE_TORCH_TO_CU130 = True   # if the driver is >= 580, upgrade PyTorch to cu130 for the fast int8 kernels
COMPUTE_DTYPE = "auto"          # "auto" (bf16, safe) | "fp16" (faster on T4, may give black images) | "fp32" (slowest)
DOWNLOAD_EDIT = True            # Fibo 2.0 Edit Turbo for the "Edit" tab (+8.4 GB)

# Model files: (Hugging Face repo, revision, path inside the repo, size in bytes, SHA256)
FIBO_REPO, FIBO_REVISION = "Comfy-Org/Fibo-2.0", "b2bffaf57d48825a6989b986f24d5cfb8c71231b"
MODEL_FILES = [
    ("diffusion_models/fibo2_dev_int8_convrot.safetensors", 8408562474,
     "063dd397395809fd22234c6bd32932cfcb22619f4d2282bdbf1dc58c406fe325"),
    ("text_encoders/qwen3vl_4b_int8_convrot.safetensors", 4858559240,
     "32de113cbf8282b02cbff0794b637b9d6676da09314a1db5a30bcea9bf4ee01c"),
    ("vae/flux2-vae.safetensors", 336211292,
     "868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3"),
]
EDIT_FILE = ("diffusion_models/fibo2_edit_turbo_int8_convrot.safetensors", 8408562474,
             "1f6fa7c499191f405467123f655f11762661eaa5bd479d1da8f161398ef72c3b")
if DOWNLOAD_EDIT:
    MODEL_FILES.append(EDIT_FILE)

# 2K / 4K output: Real-ESRGAN upscalers from the official GitHub releases (url, size in bytes, SHA256)
UPSCALERS = {
    "RealESRGAN_x2plus.pth": ("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
                              67061725, "49fafd45f8fd7aa8d31ab2a22d14d91b536c34494a5cfe31eb5d89c2fa266abb"),
    "RealESRGAN_x4plus.pth": ("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
                              67040989, "4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1"),
}

# ---- HYPIR: second upscaler (SIGGRAPH Asia 2025). Adds real detail instead of only sharpening; slower. ----
# Runs in its own process on GPU 1, so it never fights ComfyUI for memory or packages. ~3.7 GB more.
# If the install fails the notebook does NOT stop: the HYPIR option just stays hidden.
INSTALL_HYPIR = True
HYPIR_DTYPE = "fp16"            # "fp16" (fast on T4; falls back to bf16 by itself if a result comes out broken) | "bf16"
HYPIR_STRIDE = 384              # tile step (tiles are 512 px). 384 = ~1.9× faster; 256 = the authors' default (more overlap)
HYPIR_REPO = "https://github.com/XPixelGroup/HYPIR.git"
HYPIR_COMMIT = "b61d107c6cef38f01a93c7833558869731cfa8c1"
HYPIR_WEIGHTS = ("lxq007/HYPIR", "0954d1026903bc0dc05248910663fce2d4231430", "HYPIR_sd2.pth", 1038090392,
                 "d538a2cb925451fab1f75adfe715ac2b1c8bb12fa32a0851ba01be2347b354d6")
SD21_REPO, SD21_REVISION = "sd2-community/stable-diffusion-2-1-base", "4e63672c03103b6c636b8fb4119ba982469b2955"
SD21_FILES = [   # (path in the repo, saved as, size in bytes, SHA256). fp16 weights are saved under the names HYPIR loads.
    ("model_index.json", "model_index.json", None, None),
    ("scheduler/scheduler_config.json", "scheduler/scheduler_config.json", None, None),
    ("tokenizer/merges.txt", "tokenizer/merges.txt", None, None),
    ("tokenizer/special_tokens_map.json", "tokenizer/special_tokens_map.json", None, None),
    ("tokenizer/tokenizer_config.json", "tokenizer/tokenizer_config.json", None, None),
    ("tokenizer/vocab.json", "tokenizer/vocab.json", None, None),
    ("text_encoder/config.json", "text_encoder/config.json", None, None),
    ("text_encoder/model.fp16.safetensors", "text_encoder/model.safetensors", 680821096,
     "681c555376658c81dc273f2d737a2aeb23ddb6d1d8e5b3a7064636d359a22668"),
    ("unet/config.json", "unet/config.json", None, None),
    ("unet/diffusion_pytorch_model.fp16.safetensors", "unet/diffusion_pytorch_model.safetensors", 1731904736,
     "28ec9cf3b239c0751c201b1f6fb46b551df5862731b30a37aa1360101cb3fbab"),
    ("vae/config.json", "vae/config.json", None, None),
    ("vae/diffusion_pytorch_model.fp16.safetensors", "vae/diffusion_pytorch_model.safetensors", 167335342,
     "3e4c08995484ee61270175e9e7a072b66a6e4eeb5f0c266667fe1f45b90daf9a"),
]

# Folders
ON_KAGGLE = Path("/kaggle").exists()
WORK_ROOT = Path("/kaggle/temp/fibo_studio") if ON_KAGGLE else Path("./fibo_studio").resolve()
COMFY_HOME = WORK_ROOT / "ComfyUI"
OUTPUT_DIR = Path("/kaggle/working/outputs") if ON_KAGGLE else WORK_ROOT / "outputs"   # kept in the Output panel
LOG_FILE = WORK_ROOT / "comfyui.log"
HYPIR_HOME = WORK_ROOT / "HYPIR"          # the HYPIR code (pinned commit)
HYPIR_MODELS = WORK_ROOT / "hypir_models"  # HYPIR_sd2.pth + sd21_base/
HYPIR_LOG = WORK_ROOT / "hypir.log"
for d in (WORK_ROOT, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("✓ Settings loaded. Work folder:", WORK_ROOT)

---
## Step 2: Install ComfyUI

Checks the GPUs and disk, installs one **pinned ComfyUI commit** and its dependencies, upgrades PyTorch to **cu130** for the fast int8 kernels (when the driver allows it), and installs **HYPIR** at a pinned commit (if that fails, the notebook continues without it).

In [ ]:
# Step 2 - Environment check + install ComfyUI (pinned commit) and its dependencies
import shutil
import subprocess
import sys

def sh(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

# ---- Environment check ----
try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
except Exception:
    gpus = []
for g in gpus:
    print("GPU:", g)
if len(gpus) < 2:
    print("⚠ Fewer than 2 GPUs. Settings → Accelerator → GPU T4 x2 (it still works on one, just slower).")
print(f"Free disk in {WORK_ROOT}: {shutil.disk_usage(WORK_ROOT).free / 2**30:.0f} GB (needs ~25 GB)")

# ---- ComfyUI at one pinned commit ----
if not (COMFY_HOME / ".git").exists():
    COMFY_HOME.mkdir(parents=True, exist_ok=True)
    sh(["git", "init", "--quiet", COMFY_HOME])
    sh(["git", "-C", COMFY_HOME, "fetch", "--depth", "1", COMFYUI_REPO, COMFYUI_COMMIT])
    sh(["git", "-C", COMFY_HOME, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
head = subprocess.check_output(["git", "-C", str(COMFY_HOME), "rev-parse", "HEAD"], text=True).strip()
if head != COMFYUI_COMMIT:
    raise RuntimeError(f"ComfyUI commit mismatch: {head} != {COMFYUI_COMMIT}")
print("ComfyUI commit verified:", head[:12])

sh([sys.executable, "-m", "pip", "install", "-q", "-r", COMFY_HOME / "requirements.txt"])
# huggingface_hub 2.x conflicts with Kaggle's transformers, so it stays on 1.x. Gradio 6 for the web UI.
sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=1.5,<2.0", "gradio>=6.0,<7", "websocket-client", "psutil"])
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], check=False, capture_output=True)

# ---- PyTorch cu130 for the fast int8 kernels (needs NVIDIA driver >= 580) ----
# Fibo's int8_convrot weights run through comfy-kitchen's int8 GEMM, which only ships for CUDA 13.
def driver_major():
    try:
        return int(gpus[0].split(",")[-1].strip().split(".")[0])
    except Exception:
        return 0

def torch_in_subprocess():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__ + '|' + str(torch.version.cuda))"],
                       capture_output=True, text=True, timeout=120)
    return tuple(r.stdout.strip().split("|")) if r.returncode == 0 and "|" in r.stdout else ("?", "?")

DRIVER = driver_major()
print("NVIDIA driver major version:", DRIVER)
ver, cuda = torch_in_subprocess()
print(f"Right now: torch {ver} (CUDA {cuda})")
FAST_KERNELS = cuda.startswith("13.")
if UPGRADE_TORCH_TO_CU130 and not FAST_KERNELS:
    if DRIVER < 580:
        print("⚠ Driver < 580: cu130 cannot be used. Fast kernels stay off (generation will be slower).")
    else:
        print("Upgrading PyTorch to cu130 (from the official pytorch.org index)...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
                       capture_output=True, timeout=300)
        sh([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--index-url",
            "https://download.pytorch.org/whl/cu130", "torch", "torchvision", "torchaudio"])
        ver, cuda = torch_in_subprocess()
        FAST_KERNELS = cuda.startswith("13.")
        print(f"Result: torch {ver} (CUDA {cuda})")
        if FAST_KERNELS:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
                            "comfy-kitchen"], capture_output=True, timeout=600)
        else:
            print("⚠ The upgrade had no effect; the slow path will be used.")
print("Fast int8 kernels:", "ON ⚡" if FAST_KERNELS else "OFF (generation will be slower)")

# ---- HYPIR (optional second upscaler). If anything fails here the notebook does NOT stop. ----
HYPIR_OK = False
if INSTALL_HYPIR:
    try:
        if not (HYPIR_HOME / ".git").exists():
            HYPIR_HOME.mkdir(parents=True, exist_ok=True)
            sh(["git", "init", "--quiet", HYPIR_HOME])
            sh(["git", "-C", HYPIR_HOME, "fetch", "--depth", "1", HYPIR_REPO, HYPIR_COMMIT])
            sh(["git", "-C", HYPIR_HOME, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
        head = subprocess.check_output(["git", "-C", str(HYPIR_HOME), "rev-parse", "HEAD"], text=True).strip()
        if head != HYPIR_COMMIT:
            raise RuntimeError(f"HYPIR commit mismatch: {head}")
        # The enhancer only needs diffusers + peft. Kaggle's preinstalled torchao is too old for peft's version
        # check and HYPIR does not use it, so it is removed (same fix as the HYPIR notebook).
        sh([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.32", "peft>=0.14"])
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True)
        r = subprocess.run([sys.executable, "-c", f"import sys; sys.path.insert(0, {str(HYPIR_HOME)!r}); "
                            "from HYPIR.enhancer.sd2 import SD2Enhancer; print('ok')"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            raise RuntimeError((r.stderr or r.stdout)[-800:])
        HYPIR_OK = True
        print("✓ HYPIR installed, commit", HYPIR_COMMIT[:12])
    except Exception as e:
        print("⚠ HYPIR could not be installed, the HYPIR upscaler stays off. The rest works normally.\n ", e)

# ---- Early check: are the packages ComfyUI needs compatible with each other? ----
check = subprocess.run([sys.executable, "-c",
    "import torch, transformers, huggingface_hub, safetensors, aiohttp, gradio;"
    "print('torch', torch.__version__, '| transformers', transformers.__version__, '| gradio', gradio.__version__)"],
    capture_output=True, text=True, timeout=300)
if check.returncode != 0:
    raise RuntimeError("Package conflict (ComfyUI cannot start):\n" + (check.stderr or check.stdout)[-1500:])
print("✓ Packages compatible:", check.stdout.strip())

---
## Step 3: Download the models

About 26 GB (Fibo ~22 GB, HYPIR + Stable Diffusion 2.1 ~3.7 GB). Every file is verified by **size + SHA256**; if it does not match, it is deleted and the notebook stops. Re-running skips files that are already verified.

In [ ]:
# Step 3 - Download the models and verify them with SHA256 (resumes if interrupted)
import hashlib
import os
import time
import urllib.request
from pathlib import Path

from huggingface_hub import hf_hub_download

os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
MODELS_DIR = COMFY_HOME / "models"
GIB = 2**30

def sha256_of(path):
    h, t0 = hashlib.sha256(), time.time()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    print(f"    SHA256 computed ({path.stat().st_size / GIB:.1f} GB, {time.time() - t0:.0f} s)")
    return h.hexdigest()

def verify(path, size, sha):
    marker = path.with_name(path.name + ".verified")
    if marker.exists() and marker.read_text() == sha and path.stat().st_size == size:
        return True
    if path.stat().st_size != size or sha256_of(path) != sha:
        path.unlink()
        return False
    marker.write_text(sha)
    return True

for rel, size, sha in MODEL_FILES:
    dest = MODELS_DIR / rel
    print(f"• {rel}  ({size / GIB:.1f} GB)")
    if not (dest.exists() and verify(dest, size, sha)):
        t0 = time.time()
        hf_hub_download(FIBO_REPO, rel, revision=FIBO_REVISION, local_dir=str(MODELS_DIR))
        print(f"    downloaded in {time.time() - t0:.0f} s")
        if not verify(dest, size, sha):
            raise RuntimeError(f"{rel}: size/SHA256 does not match, file deleted. Run this cell again.")
    print("    ✓ verified")

up_dir = MODELS_DIR / "upscale_models"
up_dir.mkdir(parents=True, exist_ok=True)
for name, (url, size, sha) in UPSCALERS.items():
    dest = up_dir / name
    print(f"• upscale_models/{name}")
    if not (dest.exists() and verify(dest, size, sha)):
        urllib.request.urlretrieve(url, dest)
        if not verify(dest, size, sha):
            raise RuntimeError(f"{name}: SHA256 does not match, file deleted.")
    print("    ✓ verified")
if INSTALL_HYPIR and (HYPIR_HOME / "HYPIR").exists():
    print("• HYPIR (second upscaler)")
    repo, rev, name, size, sha = HYPIR_WEIGHTS
    dest = HYPIR_MODELS / name
    if not (dest.exists() and verify(dest, size, sha)):
        hf_hub_download(repo, name, revision=rev, local_dir=str(HYPIR_MODELS))
        if not verify(dest, size, sha):
            raise RuntimeError(f"{name}: SHA256 does not match, file deleted. Run this cell again.")
    print(f"    ✓ {name} verified")
    sd21 = HYPIR_MODELS / "sd21_base"
    for src, saved, size, sha in SD21_FILES:
        dest = sd21 / saved
        if dest.exists() and (sha is None or verify(dest, size, sha)):
            continue
        path = Path(hf_hub_download(SD21_REPO, src, revision=SD21_REVISION, local_dir=str(HYPIR_MODELS / "sd21_download")))
        dest.parent.mkdir(parents=True, exist_ok=True)
        os.replace(path, dest)
        if sha is not None and not verify(dest, size, sha):
            raise RuntimeError(f"{src}: SHA256 does not match, file deleted. Run this cell again.")
    print("    ✓ Stable Diffusion 2.1 base (fp16) verified")

print("✓ All models ready.")

---
## Step 4: Launch Fibo 2.0 Studio

Starts ComfyUI on both GPUs and opens the web page with a **public link**. While this cell runs, the studio is open; stop it with ■. Your images are saved to `/kaggle/working/outputs` (the **Output** panel on the right).

In [ ]:
# Step 4 - Launch Fibo 2.0 Studio (Gradio web UI with a public link)
# Starts ComfyUI on both GPUs, then opens the web page. While this cell runs, Studio is open; stop it with ■.
import os
import sys
import gc
import json
import time
import uuid
import random
import zipfile
import subprocess
import urllib.request
import urllib.error

URL = f"http://127.0.0.1:{PORT}"
DEV_MODEL = Path(MODEL_FILES[0][0]).name
TEXT_ENC = Path(MODEL_FILES[1][0]).name
VAE_FILE = Path(MODEL_FILES[2][0]).name
EDIT_MODEL = Path(EDIT_FILE[0]).name if (COMFY_HOME / "models" / EDIT_FILE[0]).exists() else None
HYPIR_READY = (INSTALL_HYPIR and (HYPIR_HOME / "HYPIR" / "enhancer" / "sd2.py").exists()
               and (HYPIR_MODELS / HYPIR_WEIGHTS[2]).exists() and (HYPIR_MODELS / "sd21_base" / SD21_FILES[-1][1]).exists())

# HYPIR runs in its own process (one GPU, its own packages and memory). It talks JSON lines over stdin/stdout.
HYPIR_WORKER = r'''
import json, sys, time, traceback
sys.path.insert(0, sys.argv[1])
BASE, WEIGHTS, DTYPE = sys.argv[2], sys.argv[3], sys.argv[4]
real_stdout = sys.stdout
sys.stdout = sys.stderr  # HYPIR prints; keep stdout for our messages only

def emit(**m):
    real_stdout.write(json.dumps(m) + "\n")
    real_stdout.flush()

import torch
import torchvision.transforms.functional as TF
from PIL import Image
import HYPIR.utils.common as common
from HYPIR.enhancer.sd2 import SD2Enhancer

def tiles(iterable, desc=None, disable=False, **kw):
    items = list(iterable)
    for i, item in enumerate(items, 1):
        yield item
        emit(type="progress", stage=desc or "", value=i, max=len(items))
common.tqdm = tiles  # tile progress goes to the web page

LORA = ["to_k", "to_q", "to_v", "to_out.0", "conv", "conv1", "conv2", "conv_shortcut", "conv_out",
        "proj_in", "proj_out", "ff.net.2", "ff.net.0.proj"]
DTYPES = {"fp16": torch.float16, "bf16": torch.bfloat16}
model, dtype = None, DTYPES[DTYPE]

def load():
    global model
    emit(type="status", message=f"Loading HYPIR ({'fp16' if dtype == torch.float16 else 'bf16'})")
    model = None
    torch.cuda.empty_cache()
    m = SD2Enhancer(base_model_path=BASE, weight_path=WEIGHTS, lora_modules=LORA, lora_rank=256,
                    model_t=200, coeff_t=200, device="cuda")
    m.weight_dtype = dtype
    m.init_models()
    model = m

def place(device):
    for name in ("vae", "text_encoder", "G"):
        getattr(model, name).to(device)
    torch.cuda.empty_cache()

def enhance(job):
    torch.manual_seed(job["seed"])
    torch.cuda.manual_seed_all(job["seed"])
    lq = TF.to_tensor(Image.open(job["input"]).convert("RGB")).unsqueeze(0)
    return model.enhance(lq=lq, prompt=job["prompt"], upscale=job["scale"], patch_size=512, stride=job["stride"],
                         return_type="pt")

emit(type="ready")
for line in sys.stdin:
    job = json.loads(line)
    try:
        t = time.time()
        if model is None:
            load()
        else:
            place("cuda")
        out = enhance(job)
        note = ""
        if dtype == torch.float16 and (torch.isnan(out).any() or out.mean() < 0.005):
            dtype = torch.bfloat16  # fp16 overflowed: switch to bf16 for the rest of the session
            load()
            out = enhance(job)
            note = "fp16 gave a broken result, switched to bf16"
        Image.fromarray((out[0].clamp(0, 1).permute(1, 2, 0).numpy() * 255).round().astype("uint8")).save(job["output"])
        place("cpu")  # give GPU 1 back to ComfyUI
        emit(type="done", seconds=time.time() - t, note=note)
    except Exception as e:
        traceback.print_exc()
        try:
            place("cpu")
        except Exception:
            pass
        emit(type="error", message=f"{type(e).__name__}: {e}")
'''
(WORK_ROOT / "hypir_worker.py").write_text(HYPIR_WORKER)

# ---------------------------------------------------------------------------
# 1) ComfyUI backend
# ---------------------------------------------------------------------------
def api(path, data=None, timeout=10):
    req = urllib.request.Request(f"{URL}{path}", data=None if data is None else json.dumps(data).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        body = r.read()
    return json.loads(body) if body else None


def comfy_alive():
    try:
        api("/system_stats", timeout=2)
        return True
    except Exception:
        return False


fast = subprocess.run([sys.executable, "-c", "import torch; print(torch.version.cuda)"],
                      capture_output=True, text=True).stdout.strip().startswith("13.")

# --disable-pinned-memory: Kaggle's container refuses page-locked RAM ("HostBuffer.truncate failed")
comfy_args = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
              "--output-directory", str(OUTPUT_DIR), "--temp-directory", str(WORK_ROOT / "tmp"),
              "--preview-method", "none", "--disable-auto-launch", "--disable-pinned-memory"]
if COMPUTE_DTYPE == "fp16":
    comfy_args.append("--fp16-unet")
elif COMPUTE_DTYPE == "fp32":
    comfy_args.append("--fp32-unet")

proc = globals().get("_comfy_proc")
if proc is not None and proc.poll() is None and globals().get("_comfy_args") == comfy_args and comfy_alive():
    print("ComfyUI backend already running, reusing it.")
else:
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(30)
        except subprocess.TimeoutExpired:
            proc.kill()
    subprocess.run(["pkill", "-f", f"main.py --listen 127.0.0.1 --port {PORT}"], capture_output=True)
    time.sleep(1)
    print("$", " ".join(comfy_args))
    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    _comfy_proc = subprocess.Popen(comfy_args, cwd=COMFY_HOME, env=env,
                                   stdout=open(LOG_FILE, "w"), stderr=subprocess.STDOUT)
    _comfy_args = comfy_args
    for _ in range(240):
        if comfy_alive() or _comfy_proc.poll() is not None:
            break
        time.sleep(1)
    if not comfy_alive():
        print(LOG_FILE.read_text(errors="replace")[-4000:])
        raise RuntimeError("ComfyUI did not start, see the log above.")

stats = api("/system_stats")
print(f"✓ ComfyUI {stats['system']['comfyui_version']} is running · fast int8 kernels: {'ON ⚡' if fast else 'OFF'}")
for d in stats["devices"]:
    print(f"   {d['name']}  ({d['vram_total'] / 2**30:.1f} GiB)")

if not api("/object_info/TextEncodeFibo2"):
    raise RuntimeError("This ComfyUI has no Fibo 2.0 support. Re-run Step 2.")
spec = api("/object_info/SelectModelDevice")["SelectModelDevice"]["input"]["required"]["device"]
DEVICE_OPTS = spec[1]["options"] if spec[0] == "COMBO" else spec[0]
MULTI_GPU = "gpu:1" in DEVICE_OPTS
GPU0, GPU1 = ("gpu:0", "gpu:1") if MULTI_GPU else ("default", "default")

# ---------------------------------------------------------------------------
# 2) Workflows
# ---------------------------------------------------------------------------
# Bria's own text-to-image negative caption
BRIA_NEGATIVE = '{"short_description":"","objects":[{"description":"the main subject","texture":"smooth, featureless, plastic-like, airbrushed, no fine surface detail","appearance_details":"soft blurred edges, simplified shapes, no fine detail","skin_tone_and_texture":"waxy airbrushed skin, poreless, plastic smooth"}],"background_setting":"a plain seamless studio backdrop, perfectly smooth gradient, blurred and featureless, no visible texture","lighting":{},"aesthetics":{"artifacts":["oversaturated colours","neon colour cast","overexposed highlights","lens flare","slight blur","motion blur","haze"]},"photographic_characteristics":{"depth_of_field":"shallow, heavy bokeh obscuring most of the frame","focus":"soft, slightly out of focus, lacking fine texture"},"style_medium":"","context":"","artistic_style":""}'

# The sizes Fibo 2.0 was trained at (~1 MP, multiples of 16)
ASPECTS = {
    "1:1 Square · 1024×1024": (1024, 1024),
    "16:9 Landscape · 1424×800": (1424, 800),
    "3:2 Photo · 1248×832": (1248, 832),
    "4:3 Classic · 1152×864": (1152, 864),
    "21:9 Cinematic · 1568×672": (1568, 672),
    "9:16 Story · 800×1424": (800, 1424),
    "2:3 Portrait · 832×1248": (832, 1248),
    "3:4 Portrait · 864×1152": (864, 1152),
    "Custom": None,
}
QUALITY = {"⚡ Fast · 20 steps": 20, "⚖️ Balanced · 30 steps": 30, "💎 Best · 50 steps": 50}
# Output size: Fibo renders ~1 MP natively. Real-ESRGAN (fast) or HYPIR (adds detail) takes it to 2K / 4K.
ESRGAN = {2: "RealESRGAN_x2plus.pth", 4: "RealESRGAN_x4plus.pth"}
UPSCALE = {"Native (~1 MP)": None, "2× · ~2K ⚡ Real-ESRGAN": ("esrgan", 2), "4× · ~4K ⚡ Real-ESRGAN": ("esrgan", 4)}
if HYPIR_READY:
    UPSCALE["4× · ~4K ✨ HYPIR Fast (HYPIR 2× + Real-ESRGAN 2×)"] = ("hypir_fast", 4)
    UPSCALE["4× · ~4K ✨ HYPIR Max (full 4× HYPIR, slowest)"] = ("hypir", 4)
UPSCALERS_UI = ["⚡ Real-ESRGAN · fast"] + (["✨ HYPIR Fast · HYPIR 2× + Real-ESRGAN 2×",
                                           "✨ HYPIR Max · full HYPIR (slowest)"] if HYPIR_READY else [])
HYPIR_MAX_PIXELS = 4096 * 4096   # same cap as the HYPIR notebook, keeps a T4 safe

JSON_TEMPLATE = json.dumps({
    "short_description": "A red vintage bicycle leaning against a pastel blue wall on a sunny street in Lisbon.",
    "objects": [{"description": "a red vintage bicycle with a wicker basket full of sunflowers", "location": "center"}],
    "background_setting": "a pastel blue plaster wall with a wooden door, cobblestone street",
    "lighting": {"conditions": "bright midday sun", "shadows": "hard, crisp shadows"},
    "aesthetics": {"composition": "rule of thirds", "color_scheme": "pastel blue, red, yellow"},
    "photographic_characteristics": {"camera_angle": "eye level", "lens_focal_length": "50mm"},
    "style_medium": "photograph",
    "text_render": [{"text": "PADARIA", "location": "sign above the door"}],
}, indent=2, ensure_ascii=False)

STAGES = {"unet": "Loading Fibo 2.0 model", "clip_load": "Loading text encoder", "vae_load": "Loading VAE",
          "clip": "Placing text encoder", "vae": "Placing VAE", "model": "Placing model", "cfg_split": "Copying model to both GPUs",
          "encode": "Encoding prompt", "sampler": "Sampling", "decode": "Decoding image",
          "up_load": "Loading upscaler", "upscale": "Upscaling", "src": "Loading image", "save": "Saving image"}


def _loaders(unet_name):
    return {
        "unet": {"class_type": "UNETLoader", "inputs": {"unet_name": unet_name, "weight_dtype": "default"}},
        "clip_load": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENC, "type": "fibo2"}},
        "vae_load": {"class_type": "VAELoader", "inputs": {"vae_name": VAE_FILE}},
        "clip": {"class_type": "SelectCLIPDevice", "inputs": {"clip": ["clip_load", 0], "device": GPU1}},
        "vae": {"class_type": "SelectVAEDevice", "inputs": {"vae": ["vae_load", 0], "device": GPU1}},
    }


def _save(wf, image_ref, upscaler, prefix):
    if upscaler:
        wf["up_load"] = {"class_type": "UpscaleModelLoader", "inputs": {"model_name": upscaler}}
        wf["upscale"] = {"class_type": "ImageUpscaleWithModel", "inputs": {"upscale_model": ["up_load", 0], "image": image_ref}}
        image_ref = ["upscale", 0]
    wf["save"] = {"class_type": "SaveImage", "inputs": {"images": image_ref, "filename_prefix": prefix}}
    return wf


def _finish(wf, model_ref, steps, cfg, seed, upscaler, prefix):
    wf["sampler"] = {"class_type": "KSampler", "inputs": {
        "model": model_ref, "seed": seed, "steps": steps, "cfg": cfg, "sampler_name": "euler", "scheduler": "simple",
        "positive": ["encode", 0], "negative": ["encode", 1], "latent_image": ["encode", 2], "denoise": 1.0}}
    wf["decode"] = {"class_type": "VAEDecode", "inputs": {"samples": ["sampler", 0], "vae": ["vae", 0]}}
    return _save(wf, ["decode", 0], upscaler, prefix)


def t2i_workflow(prompt, negative, width, height, steps, cfg, seed, cfg_split, upscaler):
    wf = _loaders(DEV_MODEL)
    model_ref = ["unet", 0]
    if cfg_split and MULTI_GPU and cfg != 1.0:
        # positive and negative pass run at the same time, one on each GPU
        wf["cfg_split"] = {"class_type": "MultiGPU_WorkUnits", "inputs": {"model": ["unet", 0], "max_gpus": 2}}
        model_ref = ["cfg_split", 0]
    wf["encode"] = {"class_type": "TextEncodeFibo2", "inputs": {
        "clip": ["clip", 0], "prompt": prompt, "negative_prompt": negative,
        "width": width, "height": height, "batch_size": 1}}
    return _finish(wf, model_ref, steps, cfg, seed, upscaler, "fibo2_t2i")


def edit_workflow(image_names, instruction, steps, seed, upscaler):
    wf = _loaders(EDIT_MODEL)
    wf["model"] = {"class_type": "SelectModelDevice", "inputs": {"model": ["unet", 0], "device": GPU0}}
    enc = {"clip": ["clip", 0], "prompt": instruction, "negative_prompt": "", "vae": ["vae", 0],
           "width": 0, "height": 0, "batch_size": 1}
    for i, name in enumerate(image_names, 1):
        wf[f"img{i}"] = {"class_type": "LoadImage", "inputs": {"image": name}}
        enc[f"images.image_{i}"] = [f"img{i}", 0]
    wf["encode"] = {"class_type": "TextEncodeFibo2", "inputs": enc}
    return _finish(wf, ["model", 0], steps, 1.0, seed, upscaler, "fibo2_edit")


def upscale_workflow(image_name, upscaler):
    wf = {"src": {"class_type": "LoadImage", "inputs": {"image": image_name}}}
    return _save(wf, ["src", 0], upscaler, "fibo2_upscale")


# ---------------------------------------------------------------------------
# 3) Running a workflow with live progress
# ---------------------------------------------------------------------------
import websocket
import gradio as gr
from PIL import Image, ImageOps


class Stopped(Exception):
    pass


def run_comfy(workflow, progress, label=""):
    # Node names become numeric ids (what the ComfyUI UI expects), the name is kept as the title
    ids = {k: str(i) for i, k in enumerate(workflow, 1)}
    titles = {v: k for k, v in ids.items()}
    fix = lambda v: [ids[v[0]], v[1]] if isinstance(v, list) and len(v) == 2 and v[0] in ids else v
    workflow = {ids[k]: {**n, "inputs": {a: fix(b) for a, b in n["inputs"].items()}, "_meta": {"title": k}}
                for k, n in workflow.items()}
    (WORK_ROOT / "workflows").mkdir(exist_ok=True)
    (WORK_ROOT / "workflows" / f"last_{workflow[ids['save']]['inputs']['filename_prefix']}.json").write_text(
        json.dumps(workflow, indent=1))

    client_id = uuid.uuid4().hex
    ws = websocket.create_connection(f"ws://127.0.0.1:{PORT}/ws?clientId={client_id}", timeout=10)
    ws.settimeout(5)
    try:
        try:
            prompt_id = api("/prompt", {"prompt": workflow, "client_id": client_id})["prompt_id"]
        except urllib.error.HTTPError as e:
            err = json.loads(e.read().decode() or "{}")
            details = [f"{x.get('message')} {x.get('details', '')}" for n in err.get("node_errors", {}).values()
                       for x in n.get("errors", [])]
            raise RuntimeError(err.get("error", {}).get("message", "Invalid workflow") + ": " + "; ".join(details))
        images = []
        while True:
            try:
                msg = ws.recv()
            except websocket.WebSocketTimeoutException:
                if _comfy_proc.poll() is not None:
                    raise RuntimeError("The ComfyUI backend crashed. Check the log in the ⚙️ System tab.")
                continue
            if not isinstance(msg, str):
                continue  # binary previews
            m = json.loads(msg)
            kind, d = m.get("type"), m.get("data", {})
            if d.get("prompt_id") not in (None, prompt_id):
                continue
            if kind == "executing":
                if d.get("node") is None:
                    break
                progress(0, desc=f"{label}{STAGES.get(titles.get(d['node']), 'Working')}...")
            elif kind == "progress":
                what = "Upscaling" if titles.get(d.get("node")) == "upscale" else "Sampling step"
                progress(d["value"] / max(d["max"], 1), desc=f"{label}{what} {d['value']}/{d['max']}")
            elif kind == "executed":
                images += d.get("output", {}).get("images", [])
            elif kind == "execution_error":
                raise RuntimeError(f"{d.get('node_type')}: {d.get('exception_type')}: {d.get('exception_message')}")
            elif kind == "execution_interrupted":
                raise Stopped()
            elif kind == "execution_success":
                break
    finally:
        ws.close()

    if not images:  # every node was cached: read the result from the history
        hist = api(f"/history/{prompt_id}").get(prompt_id, {})
        images = [img for out in hist.get("outputs", {}).values() for img in out.get("images", [])]
    return [str(OUTPUT_DIR / img["subfolder"] / img["filename"]) for img in images if img.get("type") == "output"]


_hypir = {"proc": None, "busy": False, "stopped": False}


def hypir_worker():
    p = _hypir["proc"]
    if p is None or p.poll() is not None:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = "1" if MULTI_GPU else "0"
        env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        p = subprocess.Popen([sys.executable, "-u", str(WORK_ROOT / "hypir_worker.py"), str(HYPIR_HOME),
                              str(HYPIR_MODELS / "sd21_base"), str(HYPIR_MODELS / HYPIR_WEIGHTS[2]), HYPIR_DTYPE],
                             stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=open(HYPIR_LOG, "a"),
                             text=True, bufsize=1, env=env)
        _hypir["proc"] = p
    return p


def hypir_run(path, scale, prompt, seed, progress, label=""):
    # ComfyUI keeps Fibo on both GPUs; unload it so HYPIR gets GPU 1 (Fibo reloads on the next generation)
    try:
        api("/free", {"unload_models": True, "free_memory": True})
    except Exception:
        pass
    w, h = size_of(path)
    scale = min(scale, (HYPIR_MAX_PIXELS / (w * h)) ** 0.5)
    out = str(OUTPUT_DIR / f"fibo2_hypir_{uuid.uuid4().hex[:8]}.png")
    p = hypir_worker()
    _hypir.update(busy=True, stopped=False)
    try:
        p.stdin.write(json.dumps({"input": path, "output": out, "scale": scale, "prompt": prompt or "", "seed": seed,
                                  "stride": HYPIR_STRIDE}) + "\n")
        p.stdin.flush()
        while True:
            line = p.stdout.readline()
            if not line:
                if _hypir["stopped"]:
                    raise Stopped()
                raise RuntimeError("The HYPIR worker stopped. Log:\n" + HYPIR_LOG.read_text(errors="replace")[-800:])
            try:
                m = json.loads(line)
            except ValueError:
                continue
            if m["type"] == "status":
                progress(0, desc=f"{label}{m['message']}...")
            elif m["type"] == "progress":
                progress(m["value"] / m["max"], desc=f"{label}HYPIR · {m['stage']} {m['value']}/{m['max']}")
            elif m["type"] == "done":
                return out, m
            elif m["type"] == "error":
                raise RuntimeError("HYPIR: " + m["message"])
    finally:
        _hypir["busy"] = False


def hypir_upscale(path, mode, scale, prompt, seed, progress, label=""):
    # "fast" at 4×: HYPIR adds the detail at 2×, Real-ESRGAN doubles it again (~4× less HYPIR work)
    if mode == "hypir_fast" and scale == 4:
        mid, info = hypir_run(path, 2, prompt, seed, progress, label)
        out = run_comfy(upscale_workflow(to_input(mid), ESRGAN[2]), progress, label)[0]
        return out, info
    return hypir_run(path, scale, prompt, seed, progress, label)


def fail_message(e):
    gc.collect()
    if isinstance(e, Stopped):
        return "⏹️ Stopped."
    msg = str(e)
    if "out of memory" in msg.lower():
        msg += "\n💡 Tip: turn off 'Use both GPUs', lower the resolution, or press 🧹 Free memory in ⚙️ System."
    return f"❌ Failed: {msg}"


def to_input(path):
    # ComfyUI reads images from its own input folder
    name = f"fibo_{uuid.uuid4().hex[:8]}.png"
    Image.open(path).convert("RGB").save(COMFY_HOME / "input" / name)
    return name


def size_of(path):
    with Image.open(path) as im:
        return im.size


# ---------------------------------------------------------------------------
# 4) Gradio handlers
# ---------------------------------------------------------------------------
def comfy_upscaler(out_size):
    # The Real-ESRGAN file to run inside ComfyUI, or None (native size or HYPIR afterwards)
    kind = UPSCALE.get(out_size)
    return ESRGAN[kind[1]] if kind and kind[0] == "esrgan" else None


def hypir_after(out_size):
    # The HYPIR mode to run after ComfyUI ("hypir" / "hypir_fast"), or None
    kind = UPSCALE.get(out_size)
    return kind[0] if kind and kind[0].startswith("hypir") else None


def generate(prompt, negative, aspect, custom_w, custom_h, quality, steps, cfg, seed, count, cfg_split, out_size,
             progress=gr.Progress()):
    if not prompt or not prompt.strip():
        return gr.update(), [], "⚠️ Please enter a prompt."
    width, height = ASPECTS.get(aspect) or (int(custom_w) // 16 * 16, int(custom_h) // 16 * 16)
    seed = random.randint(0, 2**32 - 1) if int(seed) == -1 else int(seed)
    paths, log = [], []
    t_all = time.time()
    try:
        for i in range(int(count)):
            t = time.time()
            label = f"[{i + 1}/{int(count)}] " if count > 1 else ""
            wf = t2i_workflow(prompt, negative or "", width, height, int(steps), float(cfg), seed + i, cfg_split,
                              comfy_upscaler(out_size))
            out = run_comfy(wf, progress, label)
            if hypir_after(out_size):
                out = [hypir_upscale(p, hypir_after(out_size), 4, "", seed + i, progress, label)[0] for p in out]
            paths += out
            log.append(f"  #{i + 1}  seed {seed + i}  ·  {time.time() - t:.0f}s")
    except Exception as e:
        return paths or gr.update(), paths, fail_message(e) + ("\n" + "\n".join(log) if log else "")
    w, h = size_of(paths[0])
    status = (f"✅ Done in {time.time() - t_all:.0f}s  ·  {w}×{h}  ·  {int(steps)} steps  ·  CFG {cfg}\n"
              + "\n".join(log))
    return paths, paths, status


def edit(img1, img2, img3, instruction, steps, seed, out_size, progress=gr.Progress()):
    if EDIT_MODEL is None:
        return None, None, "⚠️ Edit Turbo was not downloaded. Set DOWNLOAD_EDIT = True in Step 1, run Steps 1 and 3 again."
    sources = [p for p in (img1, img2, img3) if p]
    if not sources:
        return None, None, "⚠️ Please upload at least one image."
    if not instruction or not instruction.strip():
        return None, None, "⚠️ Please describe the edit."
    seed = random.randint(0, 2**32 - 1) if int(seed) == -1 else int(seed)
    t = time.time()
    try:
        out = run_comfy(edit_workflow([to_input(p) for p in sources], instruction, int(steps), seed,
                                      comfy_upscaler(out_size)), progress)
        if out and hypir_after(out_size):
            out = [hypir_upscale(out[0], hypir_after(out_size), 4, "", seed, progress)[0]]
    except Exception as e:
        return None, None, fail_message(e)
    if not out:
        return None, None, "❌ No image was returned."
    result, size = out[0], size_of(out[0])
    # Fibo center-crops a single source to the result's size; do the same for the before/after slider
    before = str(WORK_ROOT / "tmp" / f"before_{uuid.uuid4().hex[:8]}.jpg")
    ImageOps.fit(Image.open(sources[0]).convert("RGB"), size, Image.LANCZOS).save(before, quality=95)
    status = f"✅ Done in {time.time() - t:.0f}s  ·  {size[0]}×{size[1]}  ·  {int(steps)} steps  ·  seed {seed}"
    return result, (before, result), status


def upscale(img, upscaler, factor, prompt, seed, progress=gr.Progress()):
    if not img:
        return None, None, "⚠️ Please upload an image."
    w, h = size_of(img)
    scale = 4 if "4×" in factor else 2
    use_hypir = "HYPIR" in upscaler
    if use_hypir and w * h * 1.1 >= HYPIR_MAX_PIXELS:
        return None, None, f"⚠️ {w}×{h} is already ~4K. HYPIR's cap is 4096×4096 on T4; use a smaller image."
    if not use_hypir and w * h * scale * scale > 8192 * 8192:
        return None, None, f"⚠️ {w}×{h} × {scale} is larger than 8K. Use 2× or a smaller image."
    seed = random.randint(0, 2**32 - 1) if int(seed) == -1 else int(seed)
    t = time.time()
    note = ""
    try:
        if use_hypir:
            src = str(WORK_ROOT / "tmp" / f"src_{uuid.uuid4().hex[:8]}.png")
            Image.open(img).convert("RGB").save(src)
            result, info = hypir_upscale(src, "hypir_fast" if "Fast" in upscaler else "hypir", scale, prompt, seed, progress)
            note = f"  ·  seed {seed}" + (f"  ·  {info['note']}" if info.get("note") else "")
        else:
            result = run_comfy(upscale_workflow(to_input(img), ESRGAN[scale]), progress)[0]
    except Exception as e:
        return None, None, fail_message(e)
    size = size_of(result)
    if use_hypir and size[0] < w * scale - 8:
        note += "  ·  capped at 4096×4096 pixels for the T4"
    before = str(WORK_ROOT / "tmp" / f"before_{uuid.uuid4().hex[:8]}.jpg")
    Image.open(img).convert("RGB").resize(size, Image.LANCZOS).save(before, quality=95)
    return result, (before, result), f"✅ Done in {time.time() - t:.0f}s  ·  {w}×{h} → {size[0]}×{size[1]}{note}"


def stop():
    if _hypir["busy"] and _hypir["proc"] is not None:
        _hypir["stopped"] = True
        _hypir["proc"].kill()  # restarts on the next HYPIR job
    for path, data in (("/interrupt", {}), ("/queue", {"clear": True})):
        try:
            api(path, data)
        except Exception:
            pass


def gpu_status():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, check=True).stdout
    except Exception:
        return "nvidia-smi is not available."
    rows = ["| GPU | Name | VRAM used | Load |", "|---|---|---|---|"]
    for line in out.strip().splitlines():
        i, name, used, total, util = [x.strip() for x in line.split(",")]
        rows.append(f"| {i} | {name} | {int(used) / 1024:.1f} / {int(total) / 1024:.1f} GiB | {util}% |")
    try:
        import psutil
        vm = psutil.virtual_memory()
        rows.append(f"\n**System RAM:** {vm.used / 2**30:.1f} / {vm.total / 2**30:.1f} GiB")
    except Exception:
        pass
    return "\n".join(rows)


def free_memory():
    note = ""
    p = _hypir["proc"]
    if p is not None and p.poll() is None and not _hypir["busy"]:
        p.kill()  # HYPIR keeps ~2.6 GB RAM after its first use; it reloads on the next HYPIR job
        p.wait()
        note = " HYPIR worker closed."
    try:
        api("/free", {"unload_models": True, "free_memory": True})
    except Exception as e:
        return gpu_status() + f"\n\n❌ {e}"
    time.sleep(2)
    return gpu_status() + "\n\n🧹 Models unloaded from VRAM and RAM." + note


def server_log():
    try:
        return "".join(LOG_FILE.read_text(errors="replace").splitlines(keepends=True)[-120:])
    except Exception as e:
        return str(e)


def list_outputs():
    files = [p for p in OUTPUT_DIR.iterdir() if p.suffix.lower() in (".png", ".jpg", ".webp")]
    return [str(p) for p in sorted(files, key=lambda p: p.stat().st_mtime, reverse=True)]


def zip_outputs():
    files = list_outputs()
    if not files:
        return None
    path = WORK_ROOT / "fibo2_outputs.zip"
    with zipfile.ZipFile(path, "w") as z:
        for f in files:
            z.write(f, os.path.basename(f))
    return str(path)


# ---------------------------------------------------------------------------
# 5) Interface
# ---------------------------------------------------------------------------
(WORK_ROOT / "tmp").mkdir(exist_ok=True)
try:
    gr.close_all()
except Exception:
    pass

CSS = """
.gradio-container { max-width: 1200px !important; margin: auto !important; }
.brand-header { text-align:center; background: linear-gradient(135deg,#0f0c29 0%,#302b63 55%,#24243e 100%); padding:28px; border-radius:15px; margin-bottom:12px; box-shadow:0 10px 25px rgba(0,0,0,0.4); }
.brand-title { color:#fff; font-size:2em; font-weight:700; margin:0 0 6px 0; }
.brand-subtitle { color:#c0c0ff; font-size:1em; margin:0; font-weight:400; }
.info-chips { display:flex; justify-content:center; gap:8px; flex-wrap:wrap; margin-bottom:14px; }
.chip { background:rgba(124,92,255,0.12); color:#7c5cff; border:1px solid rgba(124,92,255,0.35); padding:5px 12px; border-radius:999px; font-size:13px; font-weight:600; }
button.primary { background: linear-gradient(135deg,#7c5cff 0%,#5b3fd9 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.stop-btn { background: linear-gradient(135deg,#ef4444 0%,#b91c1c 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.clear-btn { background: linear-gradient(135deg,#6b7280 0%,#374151 100%) !important; color:#fff !important; font-weight:600 !important; border-radius:12px !important; }
.status-box textarea { font-family: ui-monospace, monospace !important; font-size: 13px !important; }
.footer { text-align:center; padding:18px; margin-top:24px; border-top:1px solid rgba(128,128,128,0.25); color:#888; font-size:13px; }
.footer a { color:#7c5cff; }
"""

HEADER = """
<div class='brand-header'>
  <div class='brand-title'>🎨 Fibo 2.0 Image Studio</div>
  <div class='brand-subtitle'>Text-to-Image • Image Editing • 4K Upscale (Real-ESRGAN + HYPIR) — BRIA Fibo 2.0 on Kaggle 2× T4</div>
</div>
"""
chips = ["🧠 Fibo 2.0 · 8B · int8", f"⚡ Fast int8 kernels: {'ON' if fast else 'OFF'}", f"⚙️ compute: {COMPUTE_DTYPE}",
         "🖥️ Model on GPU 0 + GPU 1" if MULTI_GPU else "🖥️ Single GPU",
         "📝 Text encoder + VAE on GPU 1" if MULTI_GPU else "📝 Text encoder on GPU 0", "🔍 Up to 4K output",
         f"✨ HYPIR: {'ready' if HYPIR_READY else 'off'}"]
CHIPS = "<div class='info-chips'>" + "".join(f"<span class='chip'>{c}</span>" for c in chips) + "</div>"
FOOTER = ("<div class='footer'>Built by <a href='https://www.thebuildai.tech/' target='_blank'><b>TheBuildAI</b></a> · "
          "<a href='https://www.youtube.com/@thebuildai?sub_confirmation=1' target='_blank'>YouTube</a> · "
          "Model: Fibo 2.0 by BRIA AI (CC BY-NC 4.0, non-commercial) · Engine: ComfyUI</div>")

with gr.Blocks(title="Fibo 2.0 Image Studio — TheBuildAI") as demo:
    gr.HTML(HEADER)
    gr.HTML(CHIPS)
    last_paths = gr.State([])
    selected = gr.State(0)

    with gr.Tabs() as tabs:
        # ---------------- Text to Image ----------------
        with gr.Tab("🎨 Text to Image", id="t2i"):
            with gr.Row():
                with gr.Column(scale=5):
                    t_prompt = gr.Textbox(label="Prompt", lines=6,
                                          placeholder="Describe your image, or paste a structured JSON caption (see below).")
                    with gr.Accordion("🧩 Structured JSON prompt (recommended for full control)", open=False):
                        gr.Markdown("Fibo 2.0 was trained on **JSON captions**: objects, lighting, camera, style and "
                                    "even the exact text to render can be set field by field. Plain text works too.")
                        t_template = gr.Button("📋 Insert JSON template", size="sm")
                    t_aspect = gr.Dropdown(list(ASPECTS), value="1:1 Square · 1024×1024", label="Aspect ratio")
                    with gr.Row(visible=False) as t_custom:
                        t_w = gr.Slider(512, 2048, value=1024, step=16, label="Width")
                        t_h = gr.Slider(512, 2048, value=1024, step=16, label="Height")
                    t_quality = gr.Radio(list(QUALITY), value="⚖️ Balanced · 30 steps", label="Quality")
                    t_size = gr.Radio(list(UPSCALE), value="Native (~1 MP)", label="Output size")
                    with gr.Row():
                        t_seed = gr.Number(label="Seed (-1 for random)", value=-1, precision=0)
                        t_count = gr.Slider(1, 4, value=1, step=1, label="Number of images")
                    with gr.Accordion("🔧 Advanced", open=False):
                        with gr.Row():
                            t_steps = gr.Slider(10, 60, value=30, step=1, label="Steps")
                            t_cfg = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label="Guidance (CFG)")
                        t_neg = gr.Textbox(label="Negative prompt (Bria's default, JSON)", value=BRIA_NEGATIVE, lines=3)
                        t_split = gr.Checkbox(value=MULTI_GPU, interactive=MULTI_GPU,
                                              label="Use both GPUs (CFG split: ~2× faster, model copied to each GPU)")
                    with gr.Row():
                        t_go = gr.Button("✨ Generate", variant="primary", size="lg")
                        t_stop = gr.Button("🛑 Stop", size="lg", elem_classes="stop-btn")
                        t_clear = gr.Button("🗑️ Clear", size="lg", elem_classes="clear-btn")
                with gr.Column(scale=6):
                    t_gallery = gr.Gallery(label="Result", columns=2, height=560, object_fit="contain", preview=True)
                    t_status = gr.Textbox(label="Status", lines=4, interactive=False, elem_classes="status-box")
                    t_send = gr.Button("✏️ Send selected image to Edit", size="sm")
            gr.Examples(
                examples=[
                    ["A cozy reading nook by a rainy window, warm lamp light, a cat sleeping on a knitted blanket, photorealistic, 35mm photo", "3:2 Photo · 1248×832"],
                    ["A retro travel poster of Cappadocia with hot air balloons over fairy chimneys at sunrise, flat vintage illustration, the title \"CAPPADOCIA\" in bold art deco letters at the top", "2:3 Portrait · 832×1248"],
                    ["A lone astronaut walking across a vast red desert on Mars toward a crashed spaceship, long shadows at sunset, cinematic wide shot, anamorphic lens", "21:9 Cinematic · 1568×672"],
                    [JSON_TEMPLATE, "16:9 Landscape · 1424×800"],
                ],
                inputs=[t_prompt, t_aspect], label="Examples")

        # ---------------- Edit ----------------
        with gr.Tab("✏️ Edit Image", id="edit"):
            gr.Markdown("**Fibo 2.0 Edit Turbo**: edits in 4 steps. Upload a photo and say what to change. "
                        "Refer to extra images as `<image_2>`, `<image_3>` (e.g. *put the bag from <image_2> on the table*).")
            with gr.Row():
                with gr.Column(scale=5):
                    e_img1 = gr.Image(label="Image 1 (to edit)", type="filepath", height=320)
                    with gr.Accordion("➕ Reference images (optional)", open=False):
                        with gr.Row():
                            e_img2 = gr.Image(label="Image 2", type="filepath", height=200)
                            e_img3 = gr.Image(label="Image 3", type="filepath", height=200)
                    e_instr = gr.Textbox(label="Edit instruction", lines=3,
                                         placeholder="e.g. make it night time with neon reflections on the wet street")
                    e_size = gr.Radio(list(UPSCALE), value="Native (~1 MP)", label="Output size")
                    with gr.Row():
                        e_steps = gr.Slider(1, 12, value=4, step=1, label="Steps (4 recommended)")
                        e_seed = gr.Number(label="Seed (-1 for random)", value=-1, precision=0)
                    with gr.Row():
                        e_go = gr.Button("✨ Apply Edit", variant="primary", size="lg")
                        e_stop = gr.Button("🛑 Stop", size="lg", elem_classes="stop-btn")
                        e_clear = gr.Button("🗑️ Clear", size="lg", elem_classes="clear-btn")
                with gr.Column(scale=6):
                    e_out = gr.Image(label="Edited output", type="filepath", height=420)
                    e_status = gr.Textbox(label="Status", lines=2, interactive=False, elem_classes="status-box")
            e_slider = gr.ImageSlider(label="Before ↔ After", type="filepath", height=560)
            gr.Examples(
                examples=[["Make it winter: cover everything in fresh snow, keep the composition the same"],
                          ["Change the time to night with warm street lights and a starry sky"],
                          ["Turn the image into a watercolor painting"],
                          ["Replace the cat with a golden retriever puppy"]],
                inputs=[e_instr], label="Example instructions")

        # ---------------- Upscale ----------------
        with gr.Tab("🔍 Upscale", id="upscale"):
            gr.Markdown("Upscale **any** image to 2K / 4K. **⚡ Real-ESRGAN** sharpens what is there (seconds). "
                        "**✨ HYPIR** (SIGGRAPH Asia 2025) uses a diffusion prior to add real texture such as skin, fabric "
                        "and foliage (minutes on T4)." + ("" if HYPIR_READY else "  \n*HYPIR is off: it was not installed in Steps 2-3.*"))
            with gr.Row():
                with gr.Column(scale=5):
                    u_img = gr.Image(label="Input image", type="filepath", height=360)
                    u_upscaler = gr.Radio(UPSCALERS_UI, value=UPSCALERS_UI[0], label="Upscaler")
                    u_factor = gr.Radio(["2× · ~2K", "4× · ~4K"], value="4× · ~4K", label="Upscale factor")
                    with gr.Group(visible=False) as u_hypir_opts:
                        u_prompt = gr.Textbox(label="HYPIR prompt (optional, guides the added detail)",
                                              placeholder="e.g. sharp portrait photo, fine skin texture, detailed fabric")
                        u_seed = gr.Number(label="Seed (-1 for random)", value=-1, precision=0)
                    with gr.Row():
                        u_go = gr.Button("🔍 Upscale", variant="primary", size="lg")
                        u_stop = gr.Button("🛑 Stop", size="lg", elem_classes="stop-btn")
                        u_clear = gr.Button("🗑️ Clear", size="lg", elem_classes="clear-btn")
                with gr.Column(scale=6):
                    u_out = gr.Image(label="Upscaled output", type="filepath", height=420)
                    u_status = gr.Textbox(label="Status", lines=2, interactive=False, elem_classes="status-box")
            u_slider = gr.ImageSlider(label="Before ↔ After", type="filepath", height=560)

        # ---------------- Outputs ----------------
        with gr.Tab("🖼️ My Outputs", id="outputs"):
            with gr.Row():
                o_refresh = gr.Button("🔄 Refresh", size="sm")
                o_zip = gr.Button("📦 Download all as ZIP", size="sm")
            o_file = gr.File(label="ZIP")
            o_gallery = gr.Gallery(label=f"Saved in {OUTPUT_DIR}", columns=4, height=640, object_fit="contain")

        # ---------------- System ----------------
        with gr.Tab("⚙️ System", id="system"):
            s_gpu = gr.Markdown(gpu_status())
            with gr.Row():
                s_refresh = gr.Button("🔄 Refresh", size="sm")
                s_free = gr.Button("🧹 Free memory (unload Fibo + close HYPIR)", size="sm")
                s_logbtn = gr.Button("📜 Show server log", size="sm")
            s_log = gr.Code(label="ComfyUI log", language="shell", lines=20)

    gr.HTML(FOOTER)

    # ---- events: text to image ----
    t_aspect.change(lambda a: gr.update(visible=ASPECTS.get(a) is None), t_aspect, t_custom)
    t_quality.change(lambda q: QUALITY[q], t_quality, t_steps)
    t_template.click(lambda: JSON_TEMPLATE, outputs=t_prompt)
    t_event = t_go.click(generate, [t_prompt, t_neg, t_aspect, t_w, t_h, t_quality, t_steps, t_cfg, t_seed, t_count,
                                    t_split, t_size], [t_gallery, last_paths, t_status])
    t_stop.click(stop, cancels=[t_event])
    t_clear.click(lambda: ("", None, [], "", -1), outputs=[t_prompt, t_gallery, last_paths, t_status, t_seed])

    def on_select(evt: gr.SelectData):
        return evt.index

    t_gallery.select(on_select, None, selected)
    t_send.click(lambda paths, i: (paths[i] if paths and i < len(paths) else None, gr.Tabs(selected="edit")),
                 [last_paths, selected], [e_img1, tabs])

    # ---- events: edit ----
    e_event = e_go.click(edit, [e_img1, e_img2, e_img3, e_instr, e_steps, e_seed, e_size], [e_out, e_slider, e_status])
    e_stop.click(stop, cancels=[e_event])
    e_clear.click(lambda: (None, None, None, "", None, None, "", -1),
                  outputs=[e_img1, e_img2, e_img3, e_instr, e_out, e_slider, e_status, e_seed])

    # ---- events: upscale ----
    u_upscaler.change(lambda u: gr.update(visible="HYPIR" in u), u_upscaler, u_hypir_opts)
    u_event = u_go.click(upscale, [u_img, u_upscaler, u_factor, u_prompt, u_seed], [u_out, u_slider, u_status])
    u_stop.click(stop, cancels=[u_event])
    u_clear.click(lambda: (None, None, None, "", "", -1), outputs=[u_img, u_out, u_slider, u_status, u_prompt, u_seed])

    # ---- events: outputs & system ----
    o_refresh.click(list_outputs, outputs=o_gallery)
    o_zip.click(zip_outputs, outputs=o_file)
    s_refresh.click(gpu_status, outputs=s_gpu)
    s_free.click(free_memory, outputs=s_gpu)
    s_logbtn.click(server_log, outputs=s_log)
    demo.load(list_outputs, outputs=o_gallery)

print("Launching Gradio interface...")
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, debug=True, css=CSS, theme=gr.themes.Soft(primary_hue="violet"),
            allowed_paths=[str(OUTPUT_DIR), str(WORK_ROOT)])

---
## Step 5: Diagnostics

If something goes wrong, stop Step 4 with ■ and run this. It prints the GPU state and the useful lines of the ComfyUI log.

In [ ]:
# Step 5 - If something goes wrong: diagnostics (stop Step 4 with ■ first)
import subprocess

subprocess.run(["nvidia-smi"], check=False)

# The most useful lines for speed analysis: step time, model loading, dtype and the int8 kernels.
lines = LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines() if LOG_FILE.exists() else []
keys = ("s/it", "it/s", "Prompt executed", "got prompt", "loaded completely", "loaded partially", "lowvram",
        "Requested to load", "offload", "comfy_kitchen", "comfy-kitchen", "pytorch version", "Using ", "dtype",
        "Error", "error", "Traceback")
picked = [l for l in lines if any(k in l for k in keys)]
print(f"\n--- {len(picked)} useful lines out of {len(lines)} (last 80) ---")
print("\n".join(picked[-80:]))
print("\n--- last 40 log lines ---")
print("\n".join(lines[-40:]))

if HYPIR_LOG.exists():
    print("\n--- HYPIR worker log (last 30 lines) ---")
    print("\n".join(HYPIR_LOG.read_text(errors="replace").splitlines()[-30:]))

---
## Credits & Licenses

This notebook only **downloads** the models and tools below from their official sources and runs them inside your own Kaggle session. It does not host or redistribute them.

| Component | What it is used for | License |
|---|---|---|
| **Fibo 2.0 Dev / Edit Turbo** by BRIA AI | image generation and editing | [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/) (non-commercial; [commercial license from BRIA](https://platform.bria.ai/licensing/pricing?model=fibo-2)) |
| **Qwen3-VL-4B** (text encoder) | understands the prompt | [Apache-2.0](https://www.apache.org/licenses/LICENSE-2.0) |
| **FLUX.2 VAE** | turns latents into pixels | see [Comfy-Org/Fibo-2.0](https://huggingface.co/Comfy-Org/Fibo-2.0) |
| **Real-ESRGAN** x2plus / x4plus | fast 2K / 4K upscaling | [BSD-3-Clause](https://github.com/xinntao/Real-ESRGAN/blob/master/LICENSE) |
| **HYPIR** ([paper](https://arxiv.org/abs/2507.20590)) | detailed 4K upscaling | non-commercial only (code + weights), [license](https://github.com/XPixelGroup/HYPIR/blob/main/LICENSE) |
| **Stable Diffusion 2.1 base** | the prior HYPIR builds on | [CreativeML Open RAIL++-M](https://huggingface.co/stabilityai/stable-diffusion-2/blob/main/LICENSE-MODEL) |
| **ComfyUI** | the engine that runs the model | [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.html) |
| **Gradio** | the web page | [Apache-2.0](https://www.apache.org/licenses/LICENSE-2.0) |

**Powered by Fibo 2.0 from BRIA AI.** Images you make are your responsibility: do not present AI-generated images as real photos, only upload images you have the right to use, and follow the Kaggle Terms.

---

<div align='center'>

### 🎉 Enjoyed this notebook?

If this was helpful, please **upvote** and subscribe for more free AI tools!

  <a href='https://www.youtube.com/@thebuildai?sub_confirmation=1'>
    <img src='https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white' />
  </a>
  <a href='https://www.instagram.com/thebuildai/'>
    <img src='https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white' />
  </a>
  <a href='https://www.tiktok.com/@the.build.ai'>
    <img src='https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white' />
  </a>
  <a href='https://github.com/cafermutluozkan'>
    <img src='https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white' />
  </a>

  <p style='color: #888; margin-top: 15px;'>🎨 Fibo 2.0 Image Studio - Built by <strong><a href='https://www.thebuildai.tech/'>TheBuildAI</a></strong> 🌍</p>

</div>